# threepp - ray-traced robot sensors on a free Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markaren/threepp/blob/master/python/examples/colab/threepp_colab.ipynb)

`pip install threepp` puts a C++ 3D engine with a deferred ray-tracing Vulkan
backend into your Python. This notebook runs it on the datacenter GPU you are
looking at right now - no local install, no engine download, no assets:

1. **A ray-traced frame** - reflections, glass, emissives and volumetric fog on the free tier.
2. **The sensor rig** - metric depth, instance + class segmentation, world-space
   normals and a path-traced depth-sensor point cloud, all from one scene, with
   no annotation pass. This is the raw material a perception pipeline trains on.
3. **Physics in the loop** - PhysX rains props through the scene into an mp4;
   the labels stay pixel-perfect on every frame.
4. **Playground** - move the sun, thicken the fog, swap the glass, degrade the
   sensor. One cell, your parameters.

Before you start: **Runtime > Change runtime type > T4 GPU**. T4 and L4 carry RT
cores; A100/V100/H100 are compute-only and the setup cell will ask you to switch.

Total runtime, top to bottom: about 5 minutes.

## 1 · Setup (~2 min)

Three things stand between a stock Colab VM and a Vulkan ray tracer, none of them
threepp's: the image ships a Vulkan **userspace driver that doesn't match the
kernel module** (CUDA tolerates that, Vulkan doesn't - we stage the matched
userspace next to it), NVIDIA's Linux driver wants **a display to present into**
(a virtual one from Xvfb is enough), and GPU state should never live inside a
notebook kernel, so **every render runs in a fresh subprocess** via the `run()`
helper this cell defines.

In [ ]:
#@title GPU check · matched driver · Xvfb · pip install threepp  { display-mode: "form" }
import glob, json, os, subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"$ {cmd}\n{r.stdout}\n{r.stderr}")
    return r.stdout

# ---- the GPU lottery: RT cores or bust ------------------------------------
q = sh(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"]).strip()
gpu_name, kver = [s.strip() for s in q.split(",")[:2]]
print(f"GPU: {gpu_name}   kernel-module driver: {kver}")
if not any(t in gpu_name for t in ("T4", "L4", "RTX", "A10")):
    raise SystemExit(f"\n'{gpu_name}' has no RT cores (A100/V100/H100 are compute-only).\n"
                     "Runtime > Change runtime type > T4 GPU, then rerun this cell.")

# ---- Xvfb: the NVIDIA Linux driver wants a display to present into --------
sh("apt-get -qq update || true")
sh("apt-get -qq install -y xvfb")
if not os.path.exists("/tmp/.X99-lock"):
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1600x900x24"])
ENV = dict(os.environ, DISPLAY=":99")

# ---- Colab ships a Vulkan userspace that may not match the kernel module.
#      CUDA tolerates that; Vulkan needs an exact match -> stage the matched one.
us = sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.*.*"))
uver = us[-1].rsplit(".so.", 1)[1] if us else "none"
if uver != kver:
    stage = "/opt/nvidia-matched"
    print(f"userspace {uver} != kernel {kver}: staging the matched driver (~30 s) ...")
    if not glob.glob(f"{stage}/libGLX_nvidia.so.{kver}"):
        url = f"https://us.download.nvidia.com/tesla/{kver}/NVIDIA-Linux-x86_64-{kver}.run"
        sh(f"curl -fsSL -o /tmp/nv.run {url}")
        sh("sh /tmp/nv.run --extract-only --target /tmp/nvdrv")
        os.makedirs(stage, exist_ok=True)
        sh(f"cp /tmp/nvdrv/*.so.{kver} {stage}/")
        sh(f"ldconfig -n {stage}")                       # soname symlinks
    api = "1.3.277"
    stock = "/usr/share/vulkan/icd.d/nvidia_icd.json"
    if os.path.exists(stock):
        api = json.load(open(stock))["ICD"].get("api_version", api)
    with open(f"{stage}/nvidia_icd.json", "w") as f:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": f"{stage}/libGLX_nvidia.so.0",
                           "api_version": api}}, f)
    ENV["VK_ICD_FILENAMES"] = ENV["VK_DRIVER_FILES"] = f"{stage}/nvidia_icd.json"
    ENV["LD_LIBRARY_PATH"] = stage + ":" + ENV.get("LD_LIBRARY_PATH", "")
else:
    print("userspace matches the kernel module - no staging needed")

# ---- the engine -----------------------------------------------------------
sh(f"{sys.executable} -m pip -q install threepp")
import importlib.metadata
print("threepp", importlib.metadata.version("threepp"), "installed")

# ---- every render runs in a FRESH subprocess ------------------------------
def run(script, *args):
    p = subprocess.run([sys.executable, script, *map(str, args)],
                       env=ENV, capture_output=True, text=True)
    out = (p.stdout + "\n" + p.stderr).strip()
    print("\n".join(out.splitlines()[-25:]))
    if p.returncode != 0:
        raise RuntimeError(f"{script} exited with {p.returncode}")

with open("probe.py", "w") as f:
    f.write("\n".join([
        "import threepp as tp",
        "canvas = tp.Canvas('probe', width=96, height=96, headless=True, vsync=False)",
        "renderer = tp.VulkanRenderer(canvas)",
        "renderer.render(tp.Scene(), tp.PerspectiveCamera(60, 1.0, 0.1, 10))",
        "print('Vulkan renderer is up.')",
    ]))
run("probe.py")

## 2 · A ray-traced frame (~30 s)

A procedural HDR sky (generated in numpy, so nothing to download) lights a small
still life: a gold torus knot, a glass sphere, a ring of metals, three emissive
gems on a glossy floor, with exponential height fog in the air. The Vulkan
backend path-traces reflections and glass against the scene BVH and resolves
through a temporal pipeline, which is why the script renders ~70 warm-up frames
before saving one.

In [ ]:
%%writefile colab_scene.py
# Shared scene for this notebook: a procedural HDR sky (numpy -> Radiance .hdr,
# nothing to download) lighting a small PBR still life, tagged with class ids.
import math

import numpy as np
import threepp as tp

CLASS_NAMES = {0: "sky", 1: "props", 2: "floor", 3: "emitters"}


def _encode_rgbe(rgb):
    # linear float RGB -> Radiance RGBE bytes, shape (H, W, 4)
    rgb = np.maximum(np.asarray(rgb, np.float64), 0.0)
    m = rgb.max(axis=2)
    mask = m >= 1e-32
    safe = np.where(mask, m, 1.0)
    mant, exp = np.frexp(safe)
    scale = np.where(mask, mant * 256.0 / safe, 0.0)
    out = np.zeros(rgb.shape[:2] + (4,), np.uint8)
    for c in range(3):
        out[..., c] = np.clip(rgb[..., c] * scale, 0, 255).astype(np.uint8)
    out[..., 3] = np.where(mask, np.clip(exp + 128, 0, 255), 0).astype(np.uint8)
    return out


def make_sky(path, sun_dir, W=1024, H=512):
    # equirect HDR sky: warm horizon band, blue zenith, gaussian sun core + halo
    j = np.arange(H).reshape(H, 1)
    i = np.arange(W).reshape(1, W)
    theta = (j / H) * math.pi
    phi = (i / W) * 2 * math.pi - math.pi
    y = np.broadcast_to(np.cos(theta), (H, W))
    sin_t = np.sin(theta)
    x = sin_t * np.cos(phi)
    z = sin_t * np.sin(phi)
    t = np.clip(y, 0.0, 1.0)[..., None] ** 0.35
    sky = np.array([0.55, 0.45, 0.38]) * (1 - t) + np.array([0.05, 0.16, 0.42]) * t
    sky = sky + np.exp(-(y * y) / 0.009)[..., None] * np.array([1.0, 0.62, 0.32]) * 0.7
    sky = np.where((y < 0)[..., None], np.array([0.05, 0.045, 0.04]), sky)
    d = np.stack([x, y, z], axis=-1)
    ang = np.arccos(np.clip((d * sun_dir).sum(-1), -1, 1))
    sun = np.exp(-(ang / math.radians(1.5)) ** 2) * 60.0 + np.exp(-(ang / math.radians(9.0)) ** 2) * 2.2
    sky = sky + sun[..., None] * np.array([1.0, 0.93, 0.82])
    rgbe = _encode_rgbe(sky)
    if rgbe[0, 0, 0] == 2 and rgbe[0, 0, 1] == 2 and rgbe[0, 0, 2] < 128:
        rgbe[0, 0, 0] = 3   # first pixel must not look like an RLE run marker
    with open(path, "wb") as f:
        f.write(b"#?RADIANCE\nFORMAT=32-bit_rle_rgbe\n\n")
        f.write(b"-Y %d +X %d\n" % (H, W))
        f.write(rgbe.tobytes())
    return path


def _std(color, rough, metal=0.0):
    m = tp.MeshStandardMaterial()
    m.color = color
    m.roughness = rough
    m.metalness = metal
    return m


def build(renderer, aspect, sun_elev=15.0, sun_az=222.0, fog=0.008, glass_ior=1.5):
    # Returns (scene, camera), with class ids 1 = props, 2 = floor, 3 = emitters.
    el, az = math.radians(sun_elev), math.radians(sun_az)
    sun_dir = np.array([math.cos(el) * math.cos(az), math.sin(el), math.cos(el) * math.sin(az)])

    scene = tp.Scene()
    env = tp.RGBELoader().load(make_sky("sky.hdr", sun_dir))
    scene.environment = env     # image-based lighting
    scene.background = env      # and the backdrop
    if fog > 0:
        scene.set_fog_exp2(tp.Color(0xbcc8d6), fog)   # volumetric on Vulkan
        if hasattr(renderer, "fog_anisotropy"):
            renderer.fog_anisotropy = 0.55            # forward scatter: sun glow in the air

    scene.add(tp.HemisphereLight(0xdfeaff, 0x404a55, 0.35))
    sun = tp.DirectionalLight(0xfff2e0, 3.2)
    sun.position.set(float(sun_dir[0]) * 30, float(sun_dir[1]) * 30, float(sun_dir[2]) * 30)
    scene.add(sun)

    def tag(mesh, cls):
        if hasattr(renderer, "set_class_id"):   # Vulkan only; harmless on GL
            renderer.set_class_id(mesh, cls)

    floor = tp.Mesh(tp.PlaneGeometry(80, 80), _std(0x30343c, 0.12))
    floor.rotate_x(-math.pi / 2)
    scene.add(floor)
    tag(floor, 2)

    hero = tp.Mesh(tp.TorusKnotGeometry(0.85, 0.30), _std(0xffc24d, 0.16, 1.0))
    hero.position.set(0, 1.35, 0)
    scene.add(hero)
    tag(hero, 1)

    glass_mat = tp.MeshPhysicalMaterial()
    glass_mat.color = 0xffffff
    glass_mat.roughness = 0.04
    glass_mat.metalness = 0.0
    glass_mat.transmission = 1.0
    glass_mat.ior = glass_ior
    glass_mat.thickness = 0.8
    glass = tp.Mesh(tp.SphereGeometry(0.8, 48, 24), glass_mat)
    glass.name = "decor"   # rain.py registers named spheres as static colliders
    glass.position.set(2.6, 0.8, 1.2)
    scene.add(glass)
    tag(glass, 1)

    metals = [(0xf3f5f8, 0.05), (0xd9885a, 0.22), (0xaab2bd, 0.34),
              (0xe7a596, 0.18), (0x29b06a, 0.12), (0x3f74e0, 0.12)]
    for k, (col, rough) in enumerate(metals):
        a = 2 * math.pi * k / len(metals) + 0.5
        s = tp.Mesh(tp.SphereGeometry(0.45, 32, 16), _std(col, rough, 1.0))
        s.name = "decor"
        s.position.set(3.6 * math.cos(a), 0.45, 3.6 * math.sin(a))
        scene.add(s)
        tag(s, 1)

    for k, col in enumerate((0xff4d2e, 0x2ea8ff, 0xffd12e)):
        gem = tp.Mesh(tp.SphereGeometry(0.16, 24, 12), _std(0x101010, 0.4))
        gem.material.emissive = tp.Color(col)
        gem.material.emissive_intensity = 14.0
        a = 2.2 * k + 0.9
        gem.position.set(1.9 * math.cos(a), 0.16, 1.9 * math.sin(a))
        scene.add(gem)
        tag(gem, 3)

    camera = tp.PerspectiveCamera(46, aspect, 0.1, 200)
    camera.position.set(6.4, 3.2, 8.2)
    camera.look_at(0, 1.0, 0)
    return scene, camera


def aim(sensor, tx, ty, tz):
    # A DepthSensor senses along its local -Z (camera convention) but is a plain
    # Object3D, whose look_at points +Z at the target - so look at the mirror point.
    p = sensor.position
    sensor.look_at(2 * p.x - tx, 2 * p.y - ty, 2 * p.z - tz)


def converge(renderer, scene, camera, frames=70):
    # The Vulkan pipeline is temporal (probe GI, denoisers, TAA): let it settle
    # before reading anything back.
    for _ in range(frames):
        renderer.render(scene, camera)

In [ ]:
%%writefile hero.py
# One ray-traced frame: RT reflections, glass, emissives, volumetric fog.
import threepp as tp
import colab_scene as cs

W, H = 1280, 720
canvas = tp.Canvas("hero", width=W, height=H, headless=True, vsync=False)
renderer = tp.VulkanRenderer(canvas)
renderer.tone_mapping = tp.ToneMapping.ACESFilmic
renderer.tone_mapping_exposure = 1.05

scene, camera = cs.build(renderer, W / H)
cs.converge(renderer, scene, camera)
renderer.save_frame(scene, camera, "hero.png")
print("wrote hero.png")

In [ ]:
run("hero.py")
from IPython.display import Image as IPImage, display
display(IPImage("hero.png"))

## 3 · The sensor rig (~30 s)

The same scene, read back as data instead of pixels. The deferred renderer keeps
a full G-buffer every frame, so labels are free:

| readback | dtype | what it is |
|---|---|---|
| `render_aovs(...)` | uint8 | rgb, normals, instance segmentation, albedo tiles |
| `read_depth(...)` | float32 | metric depth in scene units |
| `read_instance_ids(...)` | uint32 | stable per-object ids (0 = sky), directly usable as masks |
| `read_class_ids(...)` | uint32 | semantic classes you assign with `renderer.set_class_id` |
| `DepthSensor.scan(...)` | float32 (N,3) | a world-space point cloud, path-traced through the same BVH |

The `DepthSensor` carries a seeded Gaussian range-noise model: same seed, same
cloud, on every machine - a dataset you can replay.

In [ ]:
%%writefile sensors.py
# The sensor rig: everything a perception pipeline trains on, from ONE scene.
import numpy as np
from PIL import Image

import threepp as tp
import colab_scene as cs

W, H = 960, 540
canvas = tp.Canvas("sensors", width=W, height=H, headless=True, vsync=False)
renderer = tp.VulkanRenderer(canvas)
renderer.tone_mapping = tp.ToneMapping.ACESFilmic
renderer.tone_mapping_exposure = 1.05

scene, camera = cs.build(renderer, W / H)
cs.converge(renderer, scene, camera)

# 8-bit visualisation tiles
aovs = renderer.render_aovs(scene, camera, ["rgb", "normals", "segmentation", "albedo"])

# lossless and metric - the trainable readbacks
depth = renderer.read_depth(scene, camera)         # (H,W) float32, scene units
ids = renderer.read_instance_ids(scene, camera)    # (H,W) uint32, 0 = sky
cls = renderer.read_class_ids(scene, camera)       # (H,W) uint32, semantic
np.save("depth.npy", depth)
fg = depth[depth < 199.0]
print(f"depth: {depth.shape} float32, foreground {fg.min():.2f}..{fg.max():.2f} m")
print(f"instances: {len(np.unique(ids)) - 1} objects + sky (stable uint32 ids)")
print("classes:", sorted(cs.CLASS_NAMES[c] for c in np.unique(cls).tolist()))

# a path-traced depth sensor: (N,3) world-space points through the same BVH
sensor = tp.DepthSensor(fov_y=62, width=320, height=240, near=0.1, far=40.0)
sensor.position.set(-5.0, 3.4, 6.0)
cs.aim(sensor, 0, 0.8, 0)
sensor.noise = tp.RangeNoiseModel(stddev=0.012, seed=7)   # seeded => replayable
pts = sensor.scan(renderer, scene)
np.save("cloud.npy", pts)
print(f"DepthSensor scan: {pts.shape[0]:,} points (12 mm noise, seed 7)")

# montage: rgb | depth | instance-seg / normals | class-seg | albedo
def gray(a, lo, hi):
    g = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
    return np.repeat((255 * (1 - g)).astype(np.uint8)[:, :, None], 3, axis=2)

pal = np.array([[16, 18, 24], [255, 179, 64], [70, 80, 96], [255, 64, 92]], np.uint8)
tiles = [aovs["rgb"], gray(depth, float(np.percentile(fg, 2)), float(np.percentile(fg, 90))), aovs["segmentation"],
         aovs["normals"], pal[np.clip(cls, 0, 3)], aovs["albedo"]]
mont = Image.new("RGB", (W * 3, H * 2))
for i, t in enumerate(tiles):
    mont.paste(Image.fromarray(t), ((i % 3) * W, (i // 3) * H))
mont.save("sensors.png")
print("wrote sensors.png")

In [ ]:
run("sensors.py")
from IPython.display import Image as IPImage, display
display(IPImage("sensors.png"))

import numpy as np
import matplotlib.pyplot as plt
pts = np.load("cloud.npy")
sel = pts[np.random.default_rng(0).choice(len(pts), size=min(20000, len(pts)), replace=False)]
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(projection="3d")
ax.scatter(sel[:, 0], sel[:, 2], sel[:, 1], s=0.4, c=sel[:, 1], cmap="viridis")
ax.set_box_aspect((1, 1, 0.35))
ax.set_title(f"path-traced DepthSensor cloud - {len(pts):,} points")
plt.show()

## 4 · Physics in the loop (~2 min)

PhysX (in the same wheel) rains two dozen props through the scene while every
frame is ray-traced to disk and ffmpeg muxes the mp4. Halfway through, the script
also grabs an rgb + segmentation pair: the labels track the tumbling objects
pixel-perfectly on every frame, because they come from the renderer's G-buffer,
not from an annotation pass.

In [ ]:
%%writefile rain.py
# PhysX rains props through the ray-traced scene; ffmpeg muxes the mp4.
# Halfway through we also save an rgb + segmentation pair: labels are per-frame
# G-buffer readbacks, so they track the motion with no annotation pass.
import math
import os
import random
import shutil
import subprocess
import tempfile

import numpy as np
from PIL import Image

import threepp as tp
import colab_scene as cs

assert tp.HAS_PHYSX, "this wheel should carry PhysX - please report this"
random.seed(11)

W, H = 960, 540
FPS, SECONDS = 30, 6
canvas = tp.Canvas("rain", width=W, height=H, headless=True, vsync=False)
renderer = tp.VulkanRenderer(canvas)
renderer.tone_mapping = tp.ToneMapping.ACESFilmic
renderer.tone_mapping_exposure = 1.05

scene, camera = cs.build(renderer, W / H)
world = tp.PhysxWorld()

# invisible static collider whose top face matches the visible floor plane
floor_col = tp.Mesh(tp.BoxGeometry(80, 1, 80), tp.MeshStandardMaterial())
floor_col.visible = False
floor_col.position.y = -0.5
scene.add(floor_col)
world.add_static(floor_col)

# the glass + metal spheres of the still life become static colliders too
for child in scene.children:
    if child.name == "decor":
        world.add_static(child)

palette = [0xE54B4B, 0x3CA0E5, 0x49C66A, 0xE5C04B, 0xA64BE5, 0xE5814B]
for _ in range(26):
    s = random.uniform(0.35, 0.7)
    mat = tp.MeshStandardMaterial()
    mat.color = random.choice(palette)
    mat.roughness = random.uniform(0.15, 0.6)
    mat.metalness = random.choice([0.0, 1.0])
    if random.random() < 0.5:
        m = tp.Mesh(tp.BoxGeometry(s, s, s), mat)
    else:
        m = tp.Mesh(tp.SphereGeometry(s * 0.6, 24, 12), mat)
    while True:   # spawn in a ring that clears the torus knot's footprint
        px, pz = random.uniform(-4.2, 4.2), random.uniform(-4.2, 4.2)
        if math.hypot(px, pz) > 1.7:
            break
    m.position.set(px, random.uniform(6, 14), pz)
    m.rotate_x(random.uniform(0, math.pi))
    m.rotate_z(random.uniform(0, math.pi))
    scene.add(m)
    world.add(m, density=300)
    renderer.set_class_id(m, 1)

cs.converge(renderer, scene, camera, 40)
outdir = tempfile.mkdtemp(prefix="rain_")
total = FPS * SECONDS
for k in range(total):
    world.step(1.0 / 60.0)
    world.step(1.0 / 60.0)
    renderer.save_frame(scene, camera, os.path.join(outdir, f"f{k:04d}.png"))
    if k == total // 2:   # mid-fall labels, straight from the G-buffer
        aovs = renderer.render_aovs(scene, camera, ["rgb", "segmentation"])
        Image.fromarray(np.concatenate([aovs["rgb"], aovs["segmentation"]], axis=1)).save("labels.png")
    if k % 30 == 0:
        print(f"  frame {k}/{total}", flush=True)

ff = shutil.which("ffmpeg")
if ff:
    subprocess.run([ff, "-y", "-loglevel", "error", "-framerate", str(FPS),
                    "-i", os.path.join(outdir, "f%04d.png"),
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", "rain.mp4"],
                   check=True)
    print(f"wrote rain.mp4 ({os.path.getsize('rain.mp4') // 1024} KB) and labels.png")
else:
    print(f"ffmpeg not found - {total} frames left in {outdir}")

In [ ]:
run("rain.py")
from base64 import b64encode
from IPython.display import HTML, Image as IPImage, display
display(HTML('<video controls autoplay muted loop width="720" src="data:video/mp4;base64,'
             + b64encode(open("rain.mp4", "rb").read()).decode() + '"></video>'))
print("mid-fall frame: rgb | instance segmentation")
display(IPImage("labels.png"))

## 5 · Playground (~30 s per run)

Every argument below reshapes the world, the lighting and the sensors together:
the sun position rebuilds the HDR sky, the fog is a real participating medium the
path tracer marches through, the glass IOR refracts the background differently,
and the noise goes straight into the depth sensor's seeded model. Edit and rerun.

Watch the rescanned point cloud as you thicken the fog: the count drops, because
the path-traced sensor loses returns to fog scatter exactly like a real lidar.

In [ ]:
%%writefile playground.py
# Your turn. Every argument reshapes the world, the lighting AND the sensors.
import sys

import numpy as np
import threepp as tp
import colab_scene as cs

P = {"sun_elev": 32.0, "sun_az": 55.0, "fog": 0.012, "glass_ior": 1.5,
     "exposure": 1.05, "noise_mm": 12.0}
for a in sys.argv[1:]:
    k, v = a.split("=")
    P[k] = float(v)

W, H = 1280, 720
canvas = tp.Canvas("playground", width=W, height=H, headless=True, vsync=False)
renderer = tp.VulkanRenderer(canvas)
renderer.tone_mapping = tp.ToneMapping.ACESFilmic
renderer.tone_mapping_exposure = P["exposure"]

scene, camera = cs.build(renderer, W / H, sun_elev=P["sun_elev"], sun_az=P["sun_az"],
                         fog=P["fog"], glass_ior=P["glass_ior"])
cs.converge(renderer, scene, camera)
renderer.save_frame(scene, camera, "playground.png")

sensor = tp.DepthSensor(fov_y=62, width=320, height=240, near=0.1, far=40.0)
sensor.position.set(-5.0, 3.4, 6.0)
cs.aim(sensor, 0, 0.8, 0)
sensor.noise = tp.RangeNoiseModel(stddev=P["noise_mm"] / 1000.0, seed=7)
pts = sensor.scan(renderer, scene)
np.save("cloud.npy", pts)
print(f"wrote playground.png; rescanned cloud: {pts.shape[0]:,} points at "
      f"{P['noise_mm']:.0f} mm noise")

In [ ]:
# a low sun behind the scene, thick fog, diamond-ish glass, a noisy sensor:
run("playground.py", "sun_elev=12", "sun_az=140", "fog=0.05",
    "glass_ior=1.31", "exposure=1.2", "noise_mm=25")
from IPython.display import Image as IPImage, display
display(IPImage("playground.png"))

# the same DepthSensor, rescanned with YOUR noise setting
import numpy as np
import matplotlib.pyplot as plt
pts = np.load("cloud.npy")
sel = pts[np.random.default_rng(0).choice(len(pts), size=min(20000, len(pts)), replace=False)]
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(projection="3d")
ax.scatter(sel[:, 0], sel[:, 2], sel[:, 1], s=0.4, c=sel[:, 1], cmap="viridis")
ax.set_box_aspect((1, 1, 0.35))
ax.set_title(f"rescanned DepthSensor cloud - {len(pts):,} points")
plt.show()

## What else is in the wheel

This notebook used one renderer and two sensors. The same `pip install threepp` also carries:

- **PhysX articulations + URDF loading** - robots, joints, motors, and a `threepp.rl` scene API.
- **Proprioception** - IMU, joint encoders, contact and force/torque sensors, all
  seeded and rate-gated for dataset generation.
- **Environments** - FFT ocean, terrain, vegetation, Gaussian splat rendering.
- **CUDA interop** - zero-copy vertex streams from Warp/torch sims into the ray tracer
  (`threepp.cuda_interop`), which is how you render a live 300k-particle fluid on this same T4.
- An **OpenGL backend** with windows, orbit controls and ImGui for local interactive use -
  the Vulkan path you just used is the headless/sensor side of the same scene graph.

Locally it is the same two lines: `pip install threepp`, swap `headless=True` for a window.

- Source, examples and issues: https://github.com/markaren/threepp
- Python examples folder: https://github.com/markaren/threepp/tree/master/python/examples

If this ran on your free tier and that surprised you, a star on the repo helps
other people find it.